[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 Hard: Sliding Window Attention

Implement **Sliding Window Attention** — used in Longformer, Mistral, etc. for efficient long-context processing.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size).

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.4 MB/s eta 0:00:00


In [2]:
import torch
import math

In [24]:
ones = torch.ones(5, 5)
torch.tril(torch.triu(ones, diagonal=0), diagonal=0)

tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]])

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE
# each token only focuses on nearby token in a certain window range
# achieves a global reception via stacking window attention
# Time Complexity: O(Nwd), compared to vanilla Attention's O(NNd)
# Space Complexity: O(Nw), while O(NN) for vanilla Attention

def sliding_window_attention(Q, K, V, window_size):
    bs, seq, d_k = K.size()
    scores = Q @ K.mT / math.sqrt(d_k)
    ones = torch.ones(seq, seq)
    mask = torch.tril(torch.triu(ones, diagonal=-window_size), diagonal=window_size) # neg for moving the diag down
    scores = scores.masked_fill(mask == 0, float('-inf'))
    att = torch.softmax(scores, dim=-1) @ V
    return att

In [33]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
print("Output shape:", out.shape)  # (1, 6, 8)

# window=0 should return V
out0 = sliding_window_attention(Q, K, V, window_size=0)
print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

Output shape: torch.Size([1, 6, 8])
window=0 == V? True


In [34]:
from torch_judge import check
check('sliding_window')


🧪 Testing: Sliding Window Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (1.5ms)
  ✅ [2/5] window_size=0 — only sees itself (0.6ms)
  ✅ [3/5] Large window equals full attention (5.0ms)
  ✅ [4/5] Distant tokens don't affect output (1.4ms)
  ✅ [5/5] Gradient flow (0.7ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (9.3ms total)
  Progress saved. Run status() to see your dashboard.

